# LeetCode 271 — Encode and Decode Strings bằng Python

Link đề: https://leetcode.com/problems/encode-and-decode-strings/ · Độ khó: **Medium** · Chủ đề: Design, Array, String

Skeleton LeetCode:

```python
class Codec:
    def encode(self, strs: List[str]) -> str:
        """Encodes a list of strings to a single string.
        """

    def decode(self, s: str) -> List[str]:
        """Decodes a single string to a list of strings.
        """


# Your Codec object will be instantiated and called as such:
# codec = Codec()
# codec.decode(codec.encode(strs))
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Mọi đáp án trong notebook đều được kiểm chứng bằng code: ta mô phỏng hai máy (hai đối tượng `Codec` **riêng biệt**) và kiểm tra `decode(encode(strs)) == strs` trên rất nhiều danh sách chuỗi, gồm các ký tự "khó" như dấu phân cách, chữ số, xuống dòng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế thuật toán mã hóa**: máy 1 (người gửi) gọi `encode(strs)` biến **danh sách chuỗi** thành **một chuỗi** gửi qua mạng; máy 2 (người nhận) gọi `decode(s)` phải khôi phục **đúng** danh sách ban đầu. Không được dùng các hàm tuần tự hóa sẵn (như `eval`, `json`...).
2. Hiểu vì sao cách ngây thơ **nối bằng một ký tự phân cách** (ví dụ `","`) **sai**: chuỗi gốc có thể chứa chính ký tự đó; ngoài ra phải phân biệt danh sách rỗng `[]` với `[""]`.
3. Nắm hai ý tưởng chuẩn: **tiền tố độ dài (length-prefix)** (`độ_dài + nội_dung`) và **thoát ký tự (escaping)**; hiểu vì sao tiền tố độ dài **không phụ thuộc** nội dung nên đúng với **mọi** tập ký tự (câu hỏi follow up).
4. Đi từ **nối bằng ký tự phân cách (sai, để thấy lỗi)** → **tiền tố độ dài cố định 4 ký tự (cách tham khảo)** → **`độ_dài#nội_dung` độ dài biến đổi** → **thoát ký tự với dấu `\`** → **khối tiêu đề chứa độ dài rồi tới phần thân** → **ký tự đánh dấu hiếm (chỉ đúng với tập ký tự giới hạn)**.
5. Nhớ rằng `encode` và `decode` chạy trên **hai đối tượng khác nhau**: **không được lưu trạng thái trong `self`** để "giúp" `decode`.

## 2. Đề bài & ví dụ

Thiết kế thuật toán **mã hóa một danh sách chuỗi thành một chuỗi**. Chuỗi mã hóa được gửi qua mạng rồi **giải mã** về danh sách chuỗi ban đầu.

Máy 1 (người gửi) có hàm:

```
string encode(vector<string> strs) { ... return encoded_string; }
```

Máy 2 (người nhận) có hàm:

```
vector<string> decode(string s) { ... return strs; }
```

Máy 1 làm: `encoded_string = encode(strs)`; Máy 2 làm: `strs2 = decode(encoded_string)`. Yêu cầu: `strs2` ở máy 2 **bằng** `strs` ở máy 1. Hãy cài đặt `encode` và `decode`.

Bạn **không** được dùng bất kỳ phương pháp tuần tự hóa (serialize) nào, như `eval`.

**Ví dụ 1:** `strs = ["Hello", "World"]` → sau `decode(encode(strs))` được `["Hello", "World"]`.

**Ví dụ 2:** `strs = [""]` → được `[""]`.

**Ràng buộc:** `1 <= strs.length <= 200`; `0 <= strs[i].length <= 200`; `strs[i]` gồm các ký tự bất kỳ trong 256 ký tự ASCII.

**Follow up:** có thể viết thuật toán tổng quát cho **bất kỳ tập ký tự nào** không?

In [1]:
from typing import List
import random

examples = [
    ["Hello", "World"],
    [""],
]
for strs in examples:
    print(f"strs={strs!r}")

strs=['Hello', 'World']
strs=['']


## 3. Kiến thức nền

### 3.1. Mô phỏng "hai máy": hai đối tượng riêng biệt

Trong bài thiết kế này, người gửi và người nhận là **hai máy khác nhau**: `encode` chạy trên một đối tượng `Codec`, còn `decode` chạy trên một đối tượng `Codec` **khác** (thậm chí ở tiến trình khác), chỉ nhận **chuỗi đã mã hóa**. Vì vậy `decode` không thể dựa vào dữ liệu mà `encode` đã lưu trong `self`. Ta viết hàm `round_trip` để mô phỏng đúng điều này.

In [2]:
def round_trip(codec_cls, strs):
    machine1 = codec_cls()
    message = machine1.encode(strs)          # chuỗi được "gửi qua mạng"
    machine2 = codec_cls()                    # máy 2 là đối tượng mới, không biết gì ngoài message
    return message, machine2.decode(message)


print("ví dụ hàm round_trip sẽ dùng ở các cách bên dưới")

ví dụ hàm round_trip sẽ dùng ở các cách bên dưới


### 3.2. Định dạng chuỗi có độ rộng cố định: `"{:4}".format(n)` và `f"{n:04d}"`

`"{:4}".format(5)` cho chuỗi **4 ký tự** căn lề phải: `"   5"` (3 dấu cách và `5`). `f"{n:04d}"` cho `"0005"` (đệm bằng `0`). `int()` đọc lại được cả hai (bỏ qua khoảng trắng, chữ số `0` đầu).

In [3]:
print(repr("{:4}".format(5)), repr(f"{5:04d}"), repr(f"{123:04d}"))
print(int("   5"), int("0005"), int("0123"))

'   5' '0005' '0123'
5 5 123


### 3.3. Cắt lát để đọc đúng đoạn

Nếu biết vị trí bắt đầu `i` và độ dài `size` của phần nội dung thì `s[i:i + size]` là nội dung đó, và vị trí tiếp theo là `i + size`. Đây là cách giải mã tiền tố độ dài.

In [4]:
msg = "   5Hello   5World"
i = 0
parts = []
while i < len(msg):
    size = int(msg[i:i + 4])            # đọc 4 ký tự độ dài
    i += 4
    parts.append(msg[i:i + size])       # lấy đúng size ký tự nội dung
    i += size
print(parts)                             # ['Hello', 'World']

['Hello', 'World']


### 3.4. `str.find` và `str.split`; vì sao `split` nguy hiểm

`s.find(sep, start)` trả vị trí đầu tiên của `sep` từ `start` (hoặc `-1`). `s.split(sep)` cắt **mọi** vị trí có `sep`: nếu **nội dung** chứa `sep` thì bị cắt sai. Dùng `find` để chỉ cắt đúng chỗ ta mong muốn (ví dụ dấu `#` đầu tiên sau chữ số).

In [5]:
print("5#ab#c".split("#"))                  # ['5', 'ab', 'c']  <- nội dung 'ab#c' bị cắt nhầm
i = "5#ab#c".find("#")
print(i, "5#ab#c"[:i], "5#ab#c"[i + 1:])     # 1 5 ab#c: chỉ cắt ở dấu # đầu tiên

['5', 'ab', 'c']
1 5 ab#c


### 3.5. Mã ký tự: `ord`, `chr`, và 256 ký tự ASCII mở rộng

`ord(c)` cho mã số của ký tự, `chr(k)` ngược lại. Đề cho các ký tự trong 256 giá trị `0..255`. Một ký tự **ngoài** khoảng này (như `chr(257)`) không thể xuất hiện trong `strs`, nên có thể dùng làm dấu phân cách "an toàn" **nếu** ta tin giả thiết này (Cách 5), nhưng không tổng quát cho mọi tập ký tự.

In [6]:
print(ord("A"), chr(65), ord("\n"), chr(0) == "\x00", len([chr(k) for k in range(256)]))
print(chr(257), ord(chr(257)))

65 A 10 True 256
ā 257


### 3.6. Danh sách rỗng và chuỗi rỗng: hai trường hợp khác nhau

`[]` (không có chuỗi nào) khác `[""]` (một chuỗi rỗng). Mã hóa phải phân biệt được: ví dụ nối bằng `","` cho cả hai kết quả `""`, không giải mã ngược được. Mã tốt luôn "ghi" sự có mặt của **mỗi** phần tử (độ dài hoặc dấu kết thúc).

In [7]:
print(repr(",".join([])), repr(",".join([""])), repr(",".join(["", ""])))     # '' '' ','
print([] == [""], [""] == ["", ""])

'' '' ','
False False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Thử cách ngây thơ: nối các chuỗi bằng dấu phẩy.

| `strs` | `encode` (nối bằng `","`) | `decode` (`split(",")`) | đúng? |
|---|---|---|---|
| `["Hello", "World"]` | `Hello,World` | `["Hello", "World"]` | đúng |
| `["a,b", "c"]` | `a,b,c` | `["a", "b", "c"]` | **sai** (3 phần tử) |
| `[""]` | `` (rỗng) | `[""]` | đúng |
| `[]` | `` (rỗng, **giống** mã của `[""]`) | `[""]` | **sai** (không phân biệt `[]` và `[""]`) |

Vấn đề: người nhận **không phân biệt được** dấu phẩy "của hệ thống" và dấu phẩy "của nội dung". Ý tưởng sửa: **nói trước độ dài** của mỗi chuỗi rồi mới viết nội dung. Khi đọc, ta biết chính xác đọc bao nhiêu ký tự, **bất kể bên trong là gì** (kể cả dấu phẩy, chữ số hay xuống dòng):

```
["a,b", "c"]  ->  "   3a,b   1c"
                    ^^^^ độ dài 3, nội dung "a,b" (3 ký tự)
                              ^^^^ độ dài 1, nội dung "c"
```

Khi đọc: lấy 4 ký tự đầu → `3`; lấy 3 ký tự tiếp → `a,b`; lấy 4 ký tự → `1`; lấy 1 ký tự → `c`. Không có sự nhập nhằng.

In [8]:
def naive_encode(strs):
    return ",".join(strs)


def naive_decode(s):
    return s.split(",")


for strs in [["Hello", "World"], ["a,b", "c"], [""], []]:
    msg = naive_encode(strs)
    print(f"{strs!r:25} -> {msg!r:15} -> {naive_decode(msg)!r:20} đúng? {naive_decode(msg) == strs}")

['Hello', 'World']        -> 'Hello,World'   -> ['Hello', 'World']   đúng? True
['a,b', 'c']              -> 'a,b,c'         -> ['a', 'b', 'c']      đúng? False
['']                      -> ''              -> ['']                 đúng? True
[]                        -> ''              -> ['']                 đúng? False


## 5. Cách 1 — Brute Force / baseline: nối bằng một ký tự phân cách (sai, để thấy giới hạn)

**Ý tưởng.** Nối các chuỗi bằng một ký tự phân cách (ví dụ `","`) và `split` khi giải mã. Rất dễ viết nhưng **sai** khi nội dung chứa ký tự phân cách và không phân biệt `[]` với `[""]`. Ta dùng nó để **chỉ ra ca kiểm thử bắt lỗi**; từ Cách 2 trở đi mới là các cách đúng.

In [9]:
class CodecNaive:
    def encode(self, strs: list[str]) -> str:
        return ",".join(strs)

    def decode(self, s: str) -> list[str]:
        return s.split(",")


def round_trip_ok(codec_cls, strs):
    _, back = round_trip(codec_cls, strs)
    return back == strs


for strs in [["Hello", "World"], ["a,b", "c"], [""], []]:
    print(f"{strs!r:22} -> round trip đúng? {round_trip_ok(CodecNaive, strs)}")

['Hello', 'World']     -> round trip đúng? True
['a,b', 'c']           -> round trip đúng? False
['']                   -> round trip đúng? True
[]                     -> round trip đúng? False


**Độ phức tạp (Cách 1).** `encode` và `decode` đều `O(L)` với `L` là tổng độ dài; nhưng **sai** với các ký tự đặc biệt. Chỉ để minh họa.

**Nút thắt.** Người nhận không biết dấu phân cách nào là thật. Cần cách mã hóa mà **việc tách không phụ thuộc nội dung**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tiền tố độ dài (length-prefix).** Trước mỗi chuỗi `s` ghi **độ dài của nó**. Khi giải mã, đọc độ dài `n` rồi đọc **đúng `n` ký tự** làm chuỗi; vị trí tiếp theo ngay sau đó là độ dài của chuỗi kế tiếp. Vì ta **đếm** ký tự chứ không **tìm** ký tự phân cách, nội dung có thể chứa bất cứ thứ gì (dấu phẩy, chữ số, `#`, xuống dòng, ký tự mã `0`...).

Một phần quan trọng là **độ dài phải đọc được không nhập nhằng**:

- **độ rộng cố định** (ví dụ 4 ký tự): luôn đọc đúng 4 ký tự đầu; hợp lệ khi độ dài tối đa `< 10⁴` (đề: ≤ 200);
- **kết thúc bằng một dấu hiệu** (ví dụ `độ_dài#`): chữ số của độ dài không chứa `#`, nên dấu `#` **đầu tiên** sau vị trí hiện tại chắc chắn là dấu kết thúc độ dài (dù nội dung có chứa `#`).

Mỗi chuỗi (kể cả chuỗi rỗng, độ dài `0`) đều có một tiền tố, nên `[]` (không tiền tố nào) khác `[""]` (một tiền tố `0`).

In [10]:
def length_prefix_demo(strs):
    return "".join(f"{len(s):04d}" + s for s in strs)


for strs in [["Hello", "World"], ["a,b", "c"], [""], [], ["", ""]]:
    print(f"{strs!r:22} -> {length_prefix_demo(strs)!r}")

['Hello', 'World']     -> '0005Hello0005World'
['a,b', 'c']           -> '0003a,b0001c'
['']                   -> '0000'
[]                     -> ''
['', '']               -> '00000000'


## 7. Các cách tiếp theo

### Cách 2 — Tiền tố độ dài cố định 4 ký tự (cách của lời giải tham khảo)

**Giải thích.** `encode`: với mỗi chuỗi `s`, nối `"{:4}".format(len(s)) + s`. `decode`: dùng chỉ số `i`; lặp khi `i < len(s)`: `size = int(s[i:i + 4])`, `i += 4`, lấy `s[i:i + size]`, `i += size`.

In [11]:
class CodecFixedWidth:
    def encode(self, strs: list[str]) -> str:
        """Mã hóa danh sách chuỗi thành một chuỗi."""
        return "".join("{:4}".format(len(s)) + s for s in strs)      # 4 ký tự độ dài + nội dung

    def decode(self, s: str) -> list[str]:
        """Giải mã một chuỗi về danh sách chuỗi."""
        ans = []
        i, n = 0, len(s)
        while i < n:
            size = int(s[i:i + 4])               # đọc độ dài
            i += 4
            ans.append(s[i:i + size])            # đọc đúng size ký tự nội dung
            i += size
        return ans


for strs in examples + [["a,b", "c"], []]:
    msg, back = round_trip(CodecFixedWidth, strs)
    print(f"{strs!r:22} -> {msg!r:28} -> {back!r}  đúng? {back == strs}")

['Hello', 'World']     -> '   5Hello   5World'         -> ['Hello', 'World']  đúng? True
['']                   -> '   0'                       -> ['']  đúng? True
['a,b', 'c']           -> '   3a,b   1c'               -> ['a,b', 'c']  đúng? True
[]                     -> ''                           -> []  đúng? True


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Codec:
    def encode(self, strs: List[str]) -> str:
        """Encodes a list of strings to a single string.
        """
        ans = []
        for s in strs:
            ans.append("{:4}".format(len(s)) + s)
        return "".join(ans)

    def decode(self, s: str) -> List[str]:
        """Decodes a single string to a list of strings.
        """
        ans = []
        i, n = 0, len(s)
        while i < n:
            size = int(s[i : i + 4])
            i += 4
            ans.append(s[i : i + size])
            i += size
        return ans

**Độ phức tạp Cách 2:** `encode` và `decode` đều **`O(L)`** với `L` là tổng độ dài các chuỗi (cộng `4` ký tự mỗi chuỗi); **Space `O(L)`** cho kết quả. Hạn chế: độ dài mỗi chuỗi phải `< 10⁴` (đề cho `≤ 200`).

### Cách 3 — `độ_dài#nội_dung` (độ dài biến đổi, không giới hạn)

**Giải thích.** `encode`: `f"{len(s)}#" + s`. `decode`: từ vị trí `i` dùng `s.find("#", i)` để tìm dấu `#` **đầu tiên** (kết thúc phần chữ số), `size = int(s[i:j])`, nội dung là `s[j + 1:j + 1 + size]`, `i = j + 1 + size`. Chữ số không chứa `#`, nên dấu `#` đầu tiên là đúng dù nội dung có `#`.

In [13]:
class CodecHash:
    def encode(self, strs: list[str]) -> str:
        return "".join(f"{len(s)}#{s}" for s in strs)

    def decode(self, s: str) -> list[str]:
        ans = []
        i = 0
        while i < len(s):
            j = s.find("#", i)                    # dấu '#' đầu tiên: kết thúc phần độ dài
            size = int(s[i:j])
            ans.append(s[j + 1:j + 1 + size])     # đọc đúng size ký tự, kể cả nếu chúng chứa '#'
            i = j + 1 + size
        return ans


for strs in examples + [["a#b", "12#3", "#"], []]:
    msg, back = round_trip(CodecHash, strs)
    print(f"{strs!r:26} -> {msg!r:28} -> {back!r}  đúng? {back == strs}")

['Hello', 'World']         -> '5#Hello5#World'             -> ['Hello', 'World']  đúng? True
['']                       -> '0#'                         -> ['']  đúng? True
['a#b', '12#3', '#']       -> '3#a#b4#12#31##'             -> ['a#b', '12#3', '#']  đúng? True
[]                         -> ''                           -> []  đúng? True


**Độ phức tạp Cách 3:** **Time `O(L)`**, **Space `O(L)`**. Không giới hạn độ dài chuỗi; tổng quát cho mọi tập ký tự (câu trả lời follow up).

### Cách 4 — Thoát ký tự (escaping) với dấu `\`

**Giải thích.** Chọn dấu kết thúc chuỗi (ví dụ `/`) và ký tự thoát `\`. `encode`: với mỗi ký tự `c` của mỗi chuỗi: nếu `c` là `\` hoặc `/` thì ghi `\` rồi `c`; ngược lại ghi `c`. Sau mỗi chuỗi ghi `/`. `decode`: duyệt từng ký tự: gặp `\` thì lấy ký tự kế **nguyên văn**; gặp `/` thì kết thúc một chuỗi; ngược lại thêm vào chuỗi hiện tại. Mỗi chuỗi (kể cả rỗng) đều có dấu kết thúc nên `[]` khác `[""]`.

In [14]:
class CodecEscape:
    ESC, END = "\\", "/"

    def encode(self, strs: list[str]) -> str:
        out = []
        for s in strs:
            for c in s:
                if c == self.ESC or c == self.END:
                    out.append(self.ESC)           # thoát ký tự đặc biệt
                out.append(c)
            out.append(self.END)                    # dấu kết thúc chuỗi
        return "".join(out)

    def decode(self, s: str) -> list[str]:
        ans, cur = [], []
        i = 0
        while i < len(s):
            c = s[i]
            if c == self.ESC:
                cur.append(s[i + 1])               # ký tự sau dấu thoát là nguyên văn
                i += 2
            elif c == self.END:
                ans.append("".join(cur))
                cur = []
                i += 1
            else:
                cur.append(c)
                i += 1
        return ans


for strs in examples + [["a/b", "c\\d", "\\/"], [], ["", ""]]:
    msg, back = round_trip(CodecEscape, strs)
    print(f"{strs!r:28} -> {msg!r:30} -> {back!r}  đúng? {back == strs}")

['Hello', 'World']           -> 'Hello/World/'                 -> ['Hello', 'World']  đúng? True
['']                         -> '/'                            -> ['']  đúng? True
['a/b', 'c\\d', '\\/']       -> 'a\\/b/c\\\\d/\\\\\\//'        -> ['a/b', 'c\\d', '\\/']  đúng? True
[]                           -> ''                             -> []  đúng? True
['', '']                     -> '//'                           -> ['', '']  đúng? True


**Độ phức tạp Cách 4:** **Time `O(L)`**, **Space `O(L)`**. Kích thước mã tối đa gấp đôi nội dung cộng số chuỗi.

### Cách 5 — Khối tiêu đề (header) chứa các độ dài, rồi phần thân

**Giải thích.** `encode`: `tiêu_đề = ",".join(độ dài mỗi chuỗi)`, rồi `";"`, rồi toàn bộ nội dung ghép liền. `decode`: cắt **một lần** tại dấu `;` đầu tiên (`split(";", 1)`; tiêu đề chỉ gồm chữ số và dấu phẩy nên `;` đầu tiên chắc chắn là dấu kết thúc tiêu đề); đọc các độ dài rồi chia phần thân theo các độ dài đó. Ưu điểm: các độ dài nằm gọn ở đầu.

In [15]:
class CodecHeader:
    def encode(self, strs: list[str]) -> str:
        header = ",".join(str(len(s)) for s in strs)
        return header + ";" + "".join(strs)

    def decode(self, s: str) -> list[str]:
        header, body = s.split(";", 1)               # chỉ cắt ở dấu ';' đầu tiên
        if header == "":
            return []                                  # không có chuỗi nào (khác với [""] vì tiêu đề là "0")
        ans, i = [], 0
        for size in map(int, header.split(",")):
            ans.append(body[i:i + size])
            i += size
        return ans


for strs in examples + [["a;b", "1,2", ";"], []]:
    msg, back = round_trip(CodecHeader, strs)
    print(f"{strs!r:26} -> {msg!r:26} -> {back!r}  đúng? {back == strs}")

['Hello', 'World']         -> '5,5;HelloWorld'           -> ['Hello', 'World']  đúng? True
['']                       -> '0;'                       -> ['']  đúng? True
['a;b', '1,2', ';']        -> '3,3,1;a;b1,2;'            -> ['a;b', '1,2', ';']  đúng? True
[]                         -> ';'                        -> []  đúng? True


**Độ phức tạp Cách 5:** **Time `O(L)`**, **Space `O(L)`**.

### Cách 6 — Ký tự đánh dấu hiếm (chỉ đúng với tập ký tự giới hạn)

**Giải thích.** Vì đề nói mọi ký tự thuộc 256 giá trị `0..255`, ta chọn `chr(257)` (ngoài tập đó) làm dấu kết thúc chuỗi: `encode` = mỗi chuỗi + `chr(257)`; `decode` = `split` rồi bỏ phần tử cuối rỗng. Cách này **đúng chỉ khi** giả thiết tập ký tự đúng; với tập ký tự bất kỳ (follow up) nó **sai**. Ta cài đặt để thấy giới hạn.

In [16]:
class CodecSentinel:
    END = chr(257)

    def encode(self, strs: list[str]) -> str:
        return "".join(s + self.END for s in strs)

    def decode(self, s: str) -> list[str]:
        return s.split(self.END)[:-1] if s else []


for strs in examples + [["a,b", "c"], [], ["", ""]]:
    msg, back = round_trip(CodecSentinel, strs)
    print(f"{strs!r:22} -> back {back!r}  đúng? {back == strs}")
print("với ký tự chr(257) trong dữ liệu (ngoài 256 giá trị của đề):", round_trip(CodecSentinel, ["a" + chr(257) + "b"])[1])

['Hello', 'World']     -> back ['Hello', 'World']  đúng? True
['']                   -> back ['']  đúng? True
['a,b', 'c']           -> back ['a,b', 'c']  đúng? True
[]                     -> back []  đúng? True
['', '']               -> back ['', '']  đúng? True
với ký tự chr(257) trong dữ liệu (ngoài 256 giá trị của đề): ['a', 'b']


**Độ phức tạp Cách 6:** **Time `O(L)`**, **Space `O(L)`**. Đơn giản nhưng **không tổng quát**.

> Cách 2 là cách nộp bài (theo lời giải tham khảo). Cách 3 là cách tổng quát nhất (follow up). Cách 4 là hướng khác (thoát ký tự). Cách 1 chỉ để chỉ ra lỗi.

## 8. TRACE từng bước

Ta theo dõi `encode` và `decode` của Cách 3 (`độ_dài#nội_dung`) trên `["a#b", "12#3", ""]` (nội dung chứa `#` và chữ số), rồi Cách 4 trên `["a/b", "c"]`.

In [17]:
def trace_hash(strs):
    print("strs =", strs)
    message = ""
    for s in strs:
        piece = f"{len(s)}#{s}"
        print(f"  encode {s!r}: thêm {piece!r}")
        message += piece
    print("  message =", repr(message))
    ans, i = [], 0
    while i < len(message):
        j = message.find("#", i)
        size = int(message[i:j])
        content = message[j + 1:j + 1 + size]
        print(f"  decode: i={i}, '#' đầu tiên ở j={j}, size={size}, nội dung={content!r}, i mới={j + 1 + size}")
        ans.append(content)
        i = j + 1 + size
    print("  =>", ans)
    return ans


assert trace_hash(["a#b", "12#3", ""]) == ["a#b", "12#3", ""]

strs = ['a#b', '12#3', '']
  encode 'a#b': thêm '3#a#b'
  encode '12#3': thêm '4#12#3'
  encode '': thêm '0#'
  message = '3#a#b4#12#30#'
  decode: i=0, '#' đầu tiên ở j=1, size=3, nội dung='a#b', i mới=5
  decode: i=5, '#' đầu tiên ở j=6, size=4, nội dung='12#3', i mới=11
  decode: i=11, '#' đầu tiên ở j=12, size=0, nội dung='', i mới=13
  => ['a#b', '12#3', '']


In [18]:
def trace_escape(strs):
    ESC, END = "\\", "/"
    print("strs =", strs)
    out = []
    for s in strs:
        for c in s:
            if c in (ESC, END):
                out.append(ESC)
            out.append(c)
        out.append(END)
    message = "".join(out)
    print("  message =", repr(message))
    ans, cur, i = [], [], 0
    while i < len(message):
        c = message[i]
        if c == ESC:
            print(f"  i={i}: '\\\\' thoát -> lấy nguyên văn {message[i + 1]!r}")
            cur.append(message[i + 1])
            i += 2
        elif c == END:
            print(f"  i={i}: '/' kết thúc chuỗi {''.join(cur)!r}")
            ans.append("".join(cur))
            cur = []
            i += 1
        else:
            cur.append(c)
            i += 1
    print("  =>", ans)
    return ans


assert trace_escape(["a/b", "c"]) == ["a/b", "c"]

strs = ['a/b', 'c']
  message = 'a\\/b/c/'
  i=1: '\\' thoát -> lấy nguyên văn '/'
  i=4: '/' kết thúc chuỗi 'a/b'
  i=6: '/' kết thúc chuỗi 'c'
  => ['a/b', 'c']


Bảng tóm tắt (Cách 3, `["a#b", "12#3", ""]`): mã hóa `3#a#b4#12#30#`.

| vị trí `i` | `#` đầu tiên ở `j` | `size` | nội dung | `i` mới |
|---|---|---|---|---|
| 0 | 1 | 3 | `a#b` | 5 |
| 5 | 6 | 4 | `12#3` | 11 |
| 11 | 12 | 0 | `` (rỗng) | 13 |

Dù nội dung chứa `#` và chữ số, mỗi lần ta chỉ tìm `#` **từ vị trí độ dài** (chắc chắn là dấu kết thúc phần độ dài) rồi **đếm** `size` ký tự.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (tiền tố độ dài cố định 4 ký tự). Đúng tên lớp `Codec`, đúng chữ ký `encode(self, strs: List[str]) -> str` và `decode(self, s: str) -> List[str]`; **không** lưu trạng thái trong `self`.

In [19]:
from typing import List


class Codec:
    def encode(self, strs: List[str]) -> str:
        """Encodes a list of strings to a single string.
        """
        ans = []
        for s in strs:
            ans.append("{:4}".format(len(s)) + s)        # 4 ký tự độ dài (căn phải) + nội dung
        return "".join(ans)

    def decode(self, s: str) -> List[str]:
        """Decodes a single string to a list of strings.
        """
        ans = []
        i, n = 0, len(s)
        while i < n:
            size = int(s[i:i + 4])                       # đọc độ dài
            i += 4
            ans.append(s[i:i + size])                    # đọc đúng size ký tự
            i += size
        return ans


for strs in examples:
    codec_sender, codec_receiver = Codec(), Codec()
    assert codec_receiver.decode(codec_sender.encode(strs)) == strs
print("PASS ví dụ đề cho class Codec (hai đối tượng riêng biệt)")

PASS ví dụ đề cho class Codec (hai đối tượng riêng biệt)


## 10. Vì sao thuật toán đúng

- **Giải mã duy nhất:** xét `message = P(s₁) + s₁ + P(s₂) + s₂ + ...` với `P(s)` là tiền tố độ dài (4 ký tự). Quy nạp: ở đầu mỗi vòng, `i` trỏ đúng vào đầu một tiền tố. 4 ký tự đó biểu diễn `len(s)` (hợp lệ vì `len ≤ 9999`), nên `s[i + 4 : i + 4 + len]` đúng là chuỗi gốc và `i + 4 + len` là đầu tiền tố kế tiếp. Không có bước nào phụ thuộc vào **nội dung** các chuỗi.
- **Không nhập nhằng với ký tự đặc biệt:** vì ta **đếm** ký tự thay vì **tìm** dấu phân cách, nội dung chứa dấu phẩy, chữ số, xuống dòng, ký tự mã 0... đều được đọc nguyên vẹn.
- **Phân biệt `[]` và `[""]`:** `[]` mã hóa thành `""`; `[""]` thành `"   0"` (một tiền tố độ dài 0); hai mã khác nhau. Mỗi phần tử (kể cả rỗng) đóng góp một tiền tố.
- **Mã hóa là đơn ánh:** hai danh sách khác nhau cho hai mã khác nhau (nếu cùng mã thì giải mã cho cùng một danh sách, mâu thuẫn).
- **Cách 3:** độ dài viết bằng chữ số nên không chứa `#`; `#` đầu tiên sau vị trí `i` là dấu kết thúc độ dài; phần còn lại giải mã như trên.
- **Cách 4:** dấu `\` thoát `\` và `/` nên trong dòng mã, `/` **không** thoát luôn là dấu kết thúc thật; đọc `\` rồi lấy ký tự kế nguyên văn khôi phục đúng nội dung.
- **Cách 5:** tiêu đề chỉ chứa chữ số và dấu phẩy, nên dấu `;` đầu tiên là dấu kết thúc tiêu đề; phần thân được chia theo các độ dài.
- **Không lưu trạng thái:** `decode` chỉ dùng tham số `s`; vì hai máy khác nhau, đây là điều bắt buộc.

## 11. Phân tích Time / Space Complexity

Gọi `L` là tổng độ dài các chuỗi, `k` là số chuỗi.

- **Cách 1 (nối bằng ký tự phân cách):** Time `O(L)`, Space `O(L)`; **sai** với ký tự phân cách trong nội dung.
- **Cách 2 (tiền tố độ dài cố định 4 ký tự):** `encode`, `decode` đều `O(L + k)`, Space `O(L + k)`. Giới hạn độ dài chuỗi `< 10⁴`.
- **Cách 3 (`độ_dài#nội_dung`):** `O(L + k)`; mỗi `find` quét tối đa vài chữ số (độ dài số chữ số của độ dài), tổng `O(L + k)`. Không giới hạn.
- **Cách 4 (thoát ký tự):** `O(L + k)`; mã dài tối đa `2L + k`.
- **Cách 5 (tiêu đề + thân):** `O(L + k)`.
- **Cách 6 (ký tự hiếm):** `O(L + k)`; chỉ đúng nếu `chr(257)` không xuất hiện trong dữ liệu.
- Không thể tốt hơn `O(L)` vì phải đọc/ghi mọi ký tự.

## 12. Bộ test

Với bài thiết kế mã hóa, kiểm tra cốt lõi là **round trip** (`decode(encode(strs)) == strs`) trên **hai đối tượng khác nhau**. Ta kiểm tra cả **tính đơn ánh** (hai danh sách khác nhau cho hai mã khác nhau) và dùng dữ liệu ngẫu nhiên có nhiều ký tự "khó" (dấu phân cách, chữ số, `\`, `/`, `#`, `,`, `;`, ký tự điều khiển, ký tự mã cao).

In [20]:
impls = {
    "fixed_width": CodecFixedWidth,
    "hash": CodecHash,
    "escape": CodecEscape,
    "header": CodecHeader,
    "sentinel": CodecSentinel,
    "submit": Codec,
}
GENERAL = ["fixed_width", "hash", "escape", "header", "submit"]       # đúng cho mọi ký tự (độ dài <= 9999)
ALL256 = ["fixed_width", "hash", "escape", "header", "sentinel", "submit"]   # đúng với 256 ký tự của đề

HARD_CHARS = list(",/\\#;:|0123456789abXY \n\t\r") + [chr(0), chr(1), chr(255), chr(127)]


def random_string(max_len=12, alphabet=None):
    alphabet = alphabet or HARD_CHARS
    return "".join(random.choice(alphabet) for _ in range(random.randint(0, max_len)))


def random_list(max_items=8, max_len=12, alphabet=None):
    return [random_string(max_len, alphabet) for _ in range(random.randint(0, max_items))]


for strs in examples:
    for name in ALL256:
        _, back = round_trip(impls[name], strs)
        assert back == strs, (name, strs, back)
print("PASS ví dụ của đề (mọi cách đúng; hai đối tượng riêng biệt)")

PASS ví dụ của đề (mọi cách đúng; hai đối tượng riêng biệt)


### 12.1. Các trường hợp biên

In [21]:
def check_all(strs, label, names=ALL256):
    for name in names:
        msg, back = round_trip(impls[name], strs)
        assert isinstance(msg, str), (label, name, "encode phải trả về chuỗi")
        assert back == strs, (label, name, strs, back)
    print("PASS", label)


check_all(["Hello", "World"], "ví dụ 1")
check_all([""], "ví dụ 2: một chuỗi rỗng")
check_all([], "danh sách rỗng (khác với [''])")
check_all(["", ""], "hai chuỗi rỗng")
check_all(["", "a", ""], "chuỗi rỗng ở đầu và cuối")
check_all(["a,b", "c"], "chứa dấu phẩy")
check_all(["#", "##", "1#"], "chứa dấu #")
check_all(["12#3", "0", "004"], "chứa chữ số và dấu #")
check_all(["/", "\\", "\\/", "//"], "chứa dấu thoát và dấu kết thúc của cách 4")
check_all([";", "1,2;3"], "chứa dấu ; và , của cách 5")
check_all(["a\nb", "\t", "\r\n"], "ký tự xuống dòng, tab")
check_all([chr(0), chr(255), chr(0) + chr(255)], "ký tự mã 0 và 255")
check_all(["x" * 200] * 200, "giới hạn của đề: 200 chuỗi dài 200")
check_all(["a" * 9999, "b"], "chuỗi dài 9999 (giới hạn của cách tiền tố 4 ký tự)", names=GENERAL)
check_all(["a" * 10000, "b"], "chuỗi dài 10000: chỉ các cách tổng quát", names=["hash", "escape", "header"])

PASS ví dụ 1
PASS ví dụ 2: một chuỗi rỗng
PASS danh sách rỗng (khác với [''])
PASS hai chuỗi rỗng
PASS chuỗi rỗng ở đầu và cuối
PASS chứa dấu phẩy
PASS chứa dấu #
PASS chứa chữ số và dấu #
PASS chứa dấu thoát và dấu kết thúc của cách 4
PASS chứa dấu ; và , của cách 5
PASS ký tự xuống dòng, tab
PASS ký tự mã 0 và 255
PASS giới hạn của đề: 200 chuỗi dài 200
PASS chuỗi dài 9999 (giới hạn của cách tiền tố 4 ký tự)
PASS chuỗi dài 10000: chỉ các cách tổng quát


### 12.2. Tính chất và phản ví dụ

In [22]:
random.seed(271)

# Tính chất 1 (đơn ánh): hai danh sách khác nhau -> hai mã khác nhau
for name in GENERAL:
    seen = {}
    for _ in range(3000):
        strs = random_list(4, 5, alphabet=list("a,#/\\1") + [chr(0)])
        msg = impls[name]().encode(strs)
        key = tuple(strs)
        if msg in seen:
            assert seen[msg] == key, (name, msg, seen[msg], key)
        seen[msg] = key
print("PASS tính chất: encode là đơn ánh trên 3000 danh sách ngẫu nhiên mỗi cách")

# Tính chất 2: encode không phụ thuộc trạng thái (gọi hai lần cho cùng kết quả; đối tượng khác nhau cùng kết quả)
for name in ALL256:
    strs = random_list(6, 8)
    cls = impls[name]
    a, b = cls().encode(strs), cls().encode(strs)
    obj = cls()
    assert a == b == obj.encode(strs) == obj.encode(strs)
print("PASS tính chất: encode xác định (không lưu trạng thái)")

# Phản ví dụ 1: cách nối bằng ký tự phân cách
for strs in (["a,b", "c"], [""], []):
    print(f"{strs!r:14} -> naive round trip:", round_trip(CodecNaive, strs)[1], "| đúng:", strs)
assert not round_trip_ok(CodecNaive, ["a,b", "c"])
assert round_trip_ok(CodecNaive, [""]) is True and round_trip_ok(CodecNaive, []) is False

# Phản ví dụ 2: cách ký tự hiếm sai khi dữ liệu chứa chính ký tự đó
bad = ["a" + chr(257) + "b"]
print("sentinel với chr(257) trong dữ liệu:", round_trip(CodecSentinel, bad)[1] == bad)
assert round_trip(CodecSentinel, bad)[1] != bad

PASS tính chất: encode là đơn ánh trên 3000 danh sách ngẫu nhiên mỗi cách
PASS tính chất: encode xác định (không lưu trạng thái)
['a,b', 'c']   -> naive round trip: ['a', 'b', 'c'] | đúng: ['a,b', 'c']
['']           -> naive round trip: [''] | đúng: ['']
[]             -> naive round trip: [''] | đúng: []
sentinel với chr(257) trong dữ liệu: False


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
random.seed(2026)
for trial in range(2000):
    strs = random_list(10, 15)
    for name in ALL256:
        msg, back = round_trip(impls[name], strs)
        assert back == strs, (name, strs, msg, back)

for trial in range(300):                           # chuỗi dài và nhiều chuỗi (theo giới hạn của đề)
    strs = [random_string(200, alphabet=[chr(k) for k in range(256)]) for _ in range(random.randint(1, 200))]
    for name in ALL256:
        assert round_trip(impls[name], strs)[1] == strs, name

for trial in range(300):                           # chuỗi chỉ gồm ký tự "nguy hiểm" cho từng cách
    strs = random_list(10, 10, alphabet=list("#/\\,;0123456789"))
    for name in ALL256:
        assert round_trip(impls[name], strs)[1] == strs, (name, strs)
print("PASS 2600 danh sách ngẫu nhiên: mọi cách đúng round trip (kể cả ký tự điều khiển và dấu phân cách)")

PASS 2600 danh sách ngẫu nhiên: mọi cách đúng round trip (kể cả ký tự điều khiển và dấu phân cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nối bằng ký tự phân cách và `split`

In [24]:
for strs in (["a,b", "c"], ["Hello", "World"], [""], []):
    msg, back = round_trip(CodecNaive, strs)
    print(f"{strs!r:20} -> {msg!r:12} -> {back!r:18} đúng? {back == strs}")

['a,b', 'c']         -> 'a,b,c'      -> ['a', 'b', 'c']    đúng? False
['Hello', 'World']   -> 'Hello,World' -> ['Hello', 'World'] đúng? True
['']                 -> ''           -> ['']               đúng? True
[]                   -> ''           -> ['']               đúng? False


**Vì sao sai:** nội dung có thể chứa chính ký tự phân cách (`"a,b"` thành ba phần tử khi `split`); và `[]`, `[""]` đều mã hóa thành chuỗi rỗng nên không giải mã phân biệt được. Dù chọn ký tự phân cách "lạ" cũng không an toàn vì đề cho **mọi ký tự**.

### Lỗi 2: Dùng `split("#")` thay vì chỉ cắt ở dấu `#` đầu tiên

In [25]:
class SaiSplitHash:
    def encode(self, strs):
        return "".join(f"{len(s)}#{s}" for s in strs)

    def decode(self, s):
        parts = s.split("#")                            # sai: cắt cả '#' nằm trong nội dung
        ans, i = [], 0
        try:
            while i < len(parts) - 1:
                size = int(parts[i])
                ans.append(parts[i + 1][:size])
                i += 2
        except ValueError as e:
            return f"ValueError: {e}"
        return ans


msg, back = round_trip(SaiSplitHash, ["a#b", "c"])
print(msg, "->", back, "| đúng:", ["a#b", "c"])

3#a#b1#c -> ValueError: invalid literal for int() with base 10: 'b1' | đúng: ['a#b', 'c']


**Vì sao sai:** nội dung `a#b` chứa `#` nên `split("#")` cắt thêm một phần tử; nếu nội dung chứa chữ số, `int` đọc nhầm. Đúng: dùng độ dài để **đếm** ký tự nội dung (`s[j + 1:j + 1 + size]`), chỉ dùng `find` cho `#` đầu tiên sau chữ số độ dài.

### Lỗi 3: Độ dài biến đổi nhưng không có dấu kết thúc độ dài

In [26]:
class SaiKhongDauKetThuc:
    def encode(self, strs):
        return "".join(str(len(s)) + s for s in strs)           # sai: không biết độ dài có mấy chữ số

    def decode(self, s):
        ans, i = [], 0
        while i < len(s):
            size = int(s[i])                                    # tưởng độ dài là một chữ số
            ans.append(s[i + 1:i + 1 + size])
            i += 1 + size
        return ans


strs = ["a" * 12, "b"]
msg = SaiKhongDauKetThuc().encode(strs)
print("msg =", repr(msg))
try:
    back = SaiKhongDauKetThuc().decode(msg)
    print("round trip đúng?", back == strs, back)
except ValueError as e:
    print("decode lỗi:", e)

msg = '12aaaaaaaaaaaa1b'
decode lỗi: invalid literal for int() with base 10: 'a'


**Vì sao sai:** độ dài `12` có hai chữ số, nhưng code đọc một chữ số (`1`) rồi coi `2aaaa...` là nội dung. Một tiền tố độ dài phải có **độ rộng cố định** (4 ký tự) hoặc **dấu kết thúc** (như `#`).

### Lỗi 4: Lưu dữ liệu trong `self` để `decode` dùng lại

In [27]:
class SaiCoTrangThai:
    def __init__(self):
        self.saved = None

    def encode(self, strs):
        self.saved = list(strs)               # sai: máy 2 là đối tượng khác, không có dữ liệu này
        return "x"

    def decode(self, s):
        return self.saved


one = SaiCoTrangThai()
msg = one.encode(["Hello", "World"])
print("cùng đối tượng:", one.decode(msg))
two = SaiCoTrangThai()
print("đối tượng khác (máy 2):", two.decode(msg), "| đúng: ['Hello', 'World']")

cùng đối tượng: ['Hello', 'World']
đối tượng khác (máy 2): None | đúng: ['Hello', 'World']


**Vì sao sai:** người nhận chạy trên **máy khác**, chỉ nhận được **chuỗi mã hóa**. Mọi thông tin cần để giải mã phải nằm **trong chuỗi đó**. Lỗi này chỉ lộ ra khi test bằng hai đối tượng riêng biệt (như `round_trip`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Cần ghép nhiều chuỗi **bất kỳ** thành một chuỗi rồi tách lại: **tiền tố độ dài** (đúng với mọi nội dung) hoặc **thoát ký tự**.
- Tiền tố độ dài phải **không nhập nhằng**: độ rộng cố định hoặc có dấu kết thúc (`len#`); khi đọc dùng `find` đúng chỗ, **không** `split` toàn chuỗi.
- Luôn xét `[]` và `[""]`: mỗi phần tử phải để lại "dấu vết" trong mã.
- Hàm `decode` chỉ được dùng chuỗi đã mã hóa; không lưu trạng thái trong `self` (hai máy khác nhau).
- Đây là nền tảng của nhiều giao thức thực tế (HTTP chunked, Redis RESP `$5\r\nhello`, netstring): **độ dài trước, dữ liệu sau**.
- Test mã hóa: round trip trên dữ liệu ngẫu nhiên có **ký tự phân cách, chữ số, `\`, xuống dòng**, và kiểm tra tính đơn ánh.

## 15. Biến thể của bài

1. **Mã hóa với độ dài lớn** (chuỗi dài hơn 9999): dùng `len#` thay vì độ rộng 4.
2. **Mã hóa danh sách lồng nhau** (danh sách chứa danh sách): đệ quy với tiền tố độ dài.
3. **Mã hóa cặp khóa-giá trị** (từ điển).
4. **Mã hóa nhị phân hiệu quả:** độ dài biểu diễn bằng số byte.
5. **Mã hóa danh sách số nguyên** (nén).

In [28]:
class CodecNested:
    """Biến thể 2: mã hóa danh sách lồng nhau (chuỗi hoặc danh sách) bằng tiền tố độ dài và nhãn kiểu."""

    def encode(self, obj) -> str:
        if isinstance(obj, str):
            return f"s{len(obj)}#{obj}"
        body = "".join(self.encode(x) for x in obj)
        return f"l{len(body)}#{body}"

    def decode(self, s: str):
        value, _ = self._read(s, 0)
        return value

    def _read(self, s, i):
        kind = s[i]
        j = s.find("#", i)
        size = int(s[i + 1:j])
        start = j + 1
        if kind == "s":
            return s[start:start + size], start + size
        items, pos = [], start
        while pos < start + size:
            item, pos = self._read(s, pos)
            items.append(item)
        return items, start + size


def encode_dict(d):
    """Biến thể 3."""
    flat = []
    for k, v in d.items():
        flat += [k, v]
    return CodecHash().encode(flat)


def decode_dict(msg):
    flat = CodecHash().decode(msg)
    return {flat[i]: flat[i + 1] for i in range(0, len(flat), 2)}


def encode_ints(nums):
    """Biến thể 5: danh sách số nguyên (có thể âm)."""
    return CodecHash().encode([str(x) for x in nums])


def decode_ints(msg):
    return [int(x) for x in CodecHash().decode(msg)]


nested = ["a", ["b#c", ["", "d"]], []]
codec_n = CodecNested()
assert CodecNested().decode(codec_n.encode(nested)) == nested
assert decode_dict(encode_dict({"k#1": "v,1", "": "0"})) == {"k#1": "v,1", "": "0"}
assert decode_ints(encode_ints([3, -1, 0, 10**9])) == [3, -1, 0, 10**9]
long_one = ["z" * 123456, "y"]
assert round_trip(CodecHash, long_one)[1] == long_one                # biến thể 1: chuỗi rất dài
random.seed(15)
for _ in range(300):
    d = {random_string(5): random_string(5) for _ in range(random.randint(0, 5))}
    assert decode_dict(encode_dict(d)) == d
print("PASS biến thể: chuỗi dài, danh sách lồng nhau, từ điển, số nguyên")

PASS biến thể: chuỗi dài, danh sách lồng nhau, từ điển, số nguyên


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nối bằng ký tự phân cách | `join`, `split` | `O(L)` | `O(L)` | **sai** với ký tự phân cách trong nội dung; chỉ để minh họa |
| 2. Tiền tố độ dài cố định 4 ký tự | `"{:4}".format(len) + s` | `O(L)` | `O(L)` | **cách nộp bài**; độ dài chuỗi `< 10⁴` |
| 3. `độ_dài#nội_dung` | `find("#")`, đọc `size` ký tự | `O(L)` | `O(L)` | **tổng quát**, mọi tập ký tự, mọi độ dài |
| 4. Thoát ký tự `\` | dấu kết thúc `/`, thoát `\` và `/` | `O(L)` | `O(L)` | khi muốn mã dễ đọc |
| 5. Tiêu đề + thân | các độ dài ở đầu, `split(";", 1)` | `O(L)` | `O(L)` | khi muốn đọc nhanh tất cả độ dài |
| 6. Ký tự đánh dấu hiếm | `chr(257)` làm dấu kết thúc | `O(L)` | `O(L)` | chỉ khi **chắc** tập ký tự bị giới hạn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"{:4}".format(12)`, `int("   7")`, `CodecFixedWidth().encode(["ab", ""])`, `CodecHash().encode(["x", "yz"])`, `CodecEscape().encode(["a/"])`.

**Bài 2 (tính tay).** Mã hóa tay `["hi", "a,b", ""]` bằng (a) tiền tố 4 ký tự, (b) `len#`. Sau đó giải mã lại bước từng bước (ghi `i`, `size`, nội dung).

**Bài 3 (điền TODO).** Hoàn thành `decode_todo(s)` cho cách `len#` (Cách 3).

In [29]:
def decode_todo(s):
    ans = []
    i = 0
    while i < len(s):
        # TODO 1: j = s.find("#", i)  (dấu # đầu tiên từ vị trí i)
        # TODO 2: size = int(s[i:j])
        # TODO 3: thêm s[j + 1:j + 1 + size] vào ans
        # TODO 4: i = j + 1 + size
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `class CodecHeader2` theo Cách 5 (tiêu đề các độ dài ngăn cách bằng `,`, dấu `;`, rồi thân). Kiểm tra round trip bằng hai ví dụ của đề và `[]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `class CodecBytes` mã hóa danh sách `bytes` (độ dài viết bằng chữ số thập phân, dấu `:` rồi dữ liệu, giống netstring `5:hello,`). (b) Viết `class CodecTwoLevel` mã hóa **danh sách các danh sách chuỗi** `List[List[str]]` bằng cách dùng `CodecHash` hai tầng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"{:4}"` căn phải trong 4 ký tự; `int` bỏ qua khoảng trắng. Bài 2: độ dài trước, nội dung sau, cho mỗi chuỗi.
- **Mức 2 (vừa):** Bài 1: `'  12'`, `7`, `'   2ab   0'`, `'1#x2#yz'`, `'a\\//'` (các ký tự `a`, `\`, `/`, `/`). Bài 2: (a) `'   2hi   3a,b   0'`; (b) `'2#hi3#a,b0#'`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `encode`: `",".join(str(len(s)) ...) + ";" + "".join(strs)`; `decode`: `header, body = s.split(";", 1)`; nếu `header` rỗng trả `[]`. Bài 5(a): `f"{len(b)}:".encode() + b + b","` rồi giải mã đọc đến `:`. Bài 5(b): mã hóa từng danh sách con bằng `CodecHash.encode`, rồi mã hóa danh sách các mã đó bằng `CodecHash.encode` lần nữa.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert "{:4}".format(12) == "  12"
assert int("   7") == 7
assert CodecFixedWidth().encode(["ab", ""]) == "   2ab   0"
assert CodecHash().encode(["x", "yz"]) == "1#x2#yz"
assert CodecEscape().encode(["a/"]) == "a\\//"

# Bài 2
assert CodecFixedWidth().encode(["hi", "a,b", ""]) == "   2hi   3a,b   0"
assert CodecHash().encode(["hi", "a,b", ""]) == "2#hi3#a,b0#"
assert CodecFixedWidth().decode("   2hi   3a,b   0") == ["hi", "a,b", ""]
assert CodecHash().decode("2#hi3#a,b0#") == ["hi", "a,b", ""]

# Bài 3
def decode_todo_ans(s):
    ans, i = [], 0
    while i < len(s):
        j = s.find("#", i)
        size = int(s[i:j])
        ans.append(s[j + 1:j + 1 + size])
        i = j + 1 + size
    return ans

for strs in examples + [["a#b", "12#3", ""], []]:
    assert decode_todo_ans(CodecHash().encode(strs)) == strs

# Bài 4
class CodecHeader2:
    def encode(self, strs):
        return ",".join(str(len(s)) for s in strs) + ";" + "".join(strs)

    def decode(self, s):
        header, body = s.split(";", 1)
        if not header:
            return []
        ans, i = [], 0
        for size in map(int, header.split(",")):
            ans.append(body[i:i + size])
            i += size
        return ans

for strs in examples + [[], ["a;b", ""]]:
    assert round_trip(CodecHeader2, strs)[1] == strs

# Bài 5(a), 5(b)
class CodecBytes:
    def encode(self, items):
        return b"".join(str(len(b)).encode() + b":" + b + b"," for b in items)

    def decode(self, data):
        out, i = [], 0
        while i < len(data):
            j = data.index(b":", i)
            size = int(data[i:j])
            out.append(data[j + 1:j + 1 + size])
            i = j + 1 + size + 1                 # bỏ dấu ',' kết thúc
        return out

class CodecTwoLevel:
    def encode(self, groups):
        return CodecHash().encode([CodecHash().encode(g) for g in groups])

    def decode(self, s):
        return [CodecHash().decode(x) for x in CodecHash().decode(s)]

items = [b"hello", b"", b"a:b,c", bytes(range(256))]
assert CodecBytes().decode(CodecBytes().encode(items)) == items
groups = [["a", "b#c"], [], [""], ["1#x", "2"]]
assert CodecTwoLevel().decode(CodecTwoLevel().encode(groups)) == groups
random.seed(19)
for _ in range(200):
    groups = [random_list(4, 6) for _ in range(random.randint(0, 4))]
    assert CodecTwoLevel().decode(CodecTwoLevel().encode(groups)) == groups
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tiền tố độ dài cố định 4 ký tự; không lưu trạng thái)

```python
class Codec:
    def encode(self, strs: List[str]) -> str:
        return "".join("{:4}".format(len(s)) + s for s in strs)

    def decode(self, s: str) -> List[str]:
        ans, i = [], 0
        while i < len(s):
            size = int(s[i:i + 4])
            i += 4
            ans.append(s[i:i + size])
            i += size
        return ans
```

**Checklist**

- [ ] Mã hóa **không phụ thuộc nội dung** (đếm ký tự bằng độ dài, hoặc thoát ký tự), không chỉ nối bằng một ký tự phân cách?
- [ ] Tiền tố độ dài **không nhập nhằng** (độ rộng cố định hoặc dấu kết thúc như `#`)?
- [ ] Phân biệt `[]` và `[""]` (mỗi phần tử đều để lại dấu vết)?
- [ ] `decode` chỉ dùng chuỗi đầu vào, **không** dựa vào trạng thái lưu trong `self`?
- [ ] Đọc đúng vị trí: `i += 4` rồi lấy `s[i:i + size]` rồi `i += size`?
- [ ] Đã thử: dấu phẩy, `#`, chữ số, `\`, xuống dòng, chuỗi rỗng, danh sách rỗng, 200 chuỗi dài 200?
- [ ] Đã kiểm tra round trip với **hai đối tượng** khác nhau trên dữ liệu ngẫu nhiên?
- [ ] Skeleton đúng `class Codec` / `encode(self, strs: List[str]) -> str` / `decode(self, s: str) -> List[str]`?